# GPU Performance Analysis

This tutorial will explore how to evaluate and optimize the performance of GPU-accelerated codes.
The goal is to:
- Understand the expected optimal performance of your code.
- Identify how far the actual performance deviates from the optimal.
- Explore reasons for performance gaps and practical methods to bridge them.

The material is based on a distilled version of the NHR@FAU workshop GPU Performance Engineering and designed for a two-hour event.
Feel free to check out the material on [github](https://github.com/SebastianKuckuk/gpu-performance-engineering).
Dates for upcoming public iterations for the European region are listed on the [training overview page](https://hpc.fau.de/teaching/tutorials-and-courses/) of NHR@FAU.

**Note on numbers in this notebook**:
All timings, bandwidths, and example outputs shown throughout this tutorial were measured on specific hardware - one NVIDIA A100 80 GB where not specified otherwise - part of the Alex cluster operated by NHR@FAU.\
Running the measurements and profiling on different hardware, drivers, or compiler versions can produce different absolute numbers.
What matters is not the exact values but their shape: the relative comparison between versions, the ratio between competing bottlenecks, and how close each result sits to the limits of the used hardware.\
For those that are curious: the profiles folder holds additional subfolders with example profiling data files for different GPUs (V100, A100, H100, ...).

# Introduction

Throughout this tutorial we follow the (slightly exaggerated and mildly cartoonish) journey of a fictitious PhD candidate in computer science.
Any resemblance to actual experiences is, of course, purely coincidental.

Our protagonist has just inherited a piece of legacy code: something their supervisor wrote a while back that now, at long last, needs to be published.
The catch is that it is CPU-only, and CPU-only is simply not *en vogue* anymore.
So the assignment lands on their desk: *port it to the GPU*.
After all, GPUs are all about performance and a shiny speed-up always sells.

Along the way we meet the recurring cast of their research group: the **supervisor** with strong opinions and vague technical specifics, the **wise colleague** who has seen it all before, and, doing the actual work, **our PhD candidate**.

# Our Test Application - 2D Stencil Updates

## CPU Baseline

Like any sensible engineer, our PhD candidate resists the temptation to change anything just yet.
Step one is to read the inherited code and run it exactly as handed over, establishing a baseline against which all future glory can be measured.

They soon realize that the code is basically a simple yet widely used benchmark: a 2D stencil application.
It can be regarded as a proxy application for (matrix-free) matrix-vector multiplications, which are ubiquitous in HPC applications.

A serial baseline *CPU-only* implementation can be found in [stencil-2d-base.cpp](../src/stencil-2d/stencil-2d-base.cpp).
Reviewing the implementation reveals these key points:
* The main workload is encapsulated in the `stencil2d` function.
* The application can be parameterized with command line arguments (c.f. `parseCLA_2d` in [stencil-2d-util.h](../src/stencil-2d/stencil-2d-util.h)):
  - **Data type**: `float` or `double`
  - **nx, ny**: Grid dimensions, defining total workload (`nx * ny`)
  - **nWarmUp**: Number of non-timed warm-up iterations
  - **nIt**: Number of timed iterations
* Basic diagnostic output of performance data is available via the `printStats` function in [util.h](../src/util.h).
  - Key **performance metrics** such as the sustained bandwidth are *estimated*

After reviewing the code, our PhD candidate uses the following commands to compile and execute the application:

In [ ]:
!g++ -O3 -march=native -std=c++17 ../src/stencil-2d/stencil-2d-base.cpp -o ../build/stencil-2d-base

In [ ]:
!../build/stencil-2d-base double 8192 8192 2 256

Their supervisor drops by with a hint: a serial baseline is *wonderful* for advertising maximal speed-ups after the port, but reviewers have caught on to that particular magic trick and will invariably also ask for a *parallel* CPU baseline.
Fair enough.
Our PhD candidate dusts off the memories of that one OpenMP course they took a while back and applies it to parallelize the code on the CPU first.

The updated version is available in [stencil-2d-omp-host.cpp](../src/stencil-2d/stencil-2d-omp-host.cpp), and can be compiled and executed using:

In [ ]:
!g++ -O3 -march=native -std=c++17 -fopenmp ../src/stencil-2d/stencil-2d-omp-host.cpp -o ../build/stencil-2d-omp-host

In [ ]:
!../build/stencil-2d-omp-host double 8192 8192 2 256

Depending on the parameters and the hardware, the gains may be substantial, modest, entirely absent, or, for a particularly unlucky combination of factors, even negative.

## First Attempt at GPU Acceleration

Emboldened by the CPU version, our PhD candidate recalls another nugget from that same OpenMP course: OpenMP can also target GPUs, via *target offloading*.
Same framework, new hardware: how hard could it be?
They decide to find out just how far a few extra pragmas will take them.

To offload computations to the GPU, our PhD candidate extends the code with OpenMP target offloading.
In this first attempt, the changes are limited to the `stencil2d` function.
The updated version is in [stencil-2d-omp-target-v0.cpp](../src/stencil-2d/stencil-2d-omp-target-v0.cpp).

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v0.cpp -o ../build/stencil-2d-omp-target-v0

In [ ]:
!../build/stencil-2d-omp-target-v0 double 8192 8192 2 256

The results are... not what they expected.
Our PhD candidate hoped for a triumphant speed-up; instead the GPU version is *slower* than the CPU baseline.
This is decidedly not publishable.
When they bring the bad news to their supervisor, the response is not sympathy but a question they had not thought to ask themselves: *"And what performance were you actually expecting?"*

Surprisingly, or entirely predictably for anyone who has used OpenMP target offloading before, this first GPU attempt is slower than the CPU baseline.
For the sake of the story, the underlying reason will remain a mystery for now.

There is a silver lining: performance can already be *evaluated*, which is useful for comparing different variants of the same application.
What that still does not answer, though, are exactly the questions the supervisor's raised eyebrow implied:
* Could performance be better for this particular hardware and software combination?
* If so, how can the necessary optimizations be pinpointed and applied to raise performance?

*Sustained bandwidth in GB/s, as printed by the application.*

| Version | V100 | A100 | H100 |
| --- | ---: | ---: | ---: |
| `base` (serial CPU) | 10.8 | 24.4 | 22.2 |
| `omp-host` (OpenMP CPU) | 37.8 | 25.2 | 76.9 |
| `omp-target-v0` (OpenMP GPU, naive) | 2.3 | 6.3 | 10.2 |

# Performance Models

"What performance *should* I expect?" turns out to be the right question, and our PhD candidate has no idea how to answer it.
A dim memory surfaces of the supervisor once muttering something about *performance models*.
Lacking a more precise reference, they do what anyone would: a quick round of online research.

Performance models aim at relating algorithms, their implementation, hardware characteristics and the expected performance (execution time).
One of the most straightforward ones is the *bottleneck model*.

## Bottleneck Model

The bottleneck model analyzes potential performance limiters individually, such as:
* **DRAM Bandwidth**, i.e. the speed at which data can be transferred from/to global memory.
* **Compute Throughput**, i.e. the rate at which computations can be executed.
* **Cache Bandwidth**, i.e. the speed of accessing data from different parts of the memory hierarchy, e.g. L2 cache, L1 cache, or shared memory.

After each bottleneck was modeled separately, their individual contributions are compared to find the dominant one.
This *main bottleneck* can then be used to estimate the expected performance and thereby also the expected execution time under ideal conditions - the lightspeed estimate. 

To put it into different words: application performance is generally limited by the time required to do the meaningful computations, or by getting data to and from where the compute is happening.
Whichever takes longer determines overall execution time.

# Hardware Characteristics

Suddenly those strange numbers the inherited code prints at the end make sense: they are the raw material a performance model feeds on.
To compare them against what the hardware should deliver, our PhD candidate goes looking for the specifications of the GPUs available to them.

The bottleneck model, like other performance models, applies to both CPUs and GPUs.
Since this course focuses on GPU performance, our PhD candidate examines the theoretical limits of the key GPU bottlenecks.
Relevant information can be found in official data sheets and white papers ([V100](https://images.nvidia.com/content/volta-architecture/pdf/volta-architecture-whitepaper.pdf), [A100](https://images.nvidia.com/aem-dam/en-zz/Solutions/data-center/nvidia-ampere-architecture-whitepaper.pdf), [H100](https://resources.nvidia.com/en-us-hopper-architecture/nvidia-h100-tensor-c)), system documentation, third-party resources (such as [TechPowerUp](https://www.techpowerup.com/gpu-specs/)), and obtained by running micro-benchmarks (see [below](#micro-benchmarks)).

|                             |  V100 SXM2 32 GB   | A100 SXM 40 GB \| 80 GB | H100 SXM 94 GB HBM2e  | H100 SXM5 80 GB HBM3  |
| --------------------------- | :----------------: | :---------------------: | :-------------------: | :-------------------: |
| Availability                |     Bridges-2      |      NHR@FAU Alex       |     NHR@FAU Helma     |       Bridges-2       |
| CUDA                        |        7.0         |           8.0           |          9.0          |          9.0          |
| #Cores                      | 5120<br/>(80 * 64) |   6912<br/>(108 * 64)   | 16896<br/>(132 * 128) | 16896<br/>(132 * 128) |
| FP32 Performance \[TFLOPS\] |         16         |           19            |          67           |          67           |
| FP64 Performance \[TFLOPS\] |         8          |          9.75           |          33           |          33           |
| FP64:FP32 Ratio             |        1:2         |           1:2           |          1:2          |          1:2          |
| Memory \[GB\]               |         32         |        40 \| 80         |          94           |          80           |
| Bandwidth \[GB/s\]          |        898         |      1555 \| 2039       |         2400          |         3360          |
| L2 Cache \[MB\]             |         6          |           40            |          50           |          50           |
| TDP \[W\]                   |        300         |           400           |          700          |          700          |

Comparing this first attempt against these numbers, it is painfully evident that it hits *none* of the major performance limits of the GPU: the result is not even in the same order of magnitude.
The upside is that there is clearly a lot of room to improve.

# Application Level Profiling

Armed with a performance model, our PhD candidate now knows they are (multiple) orders of magnitude away from the hardware limits, but not *why*.
Puzzled, they wander over to a more experienced colleague for advice.
The verdict is immediate and, to anyone who has done this before, entirely unsurprising: *stop guessing and start profiling.*

The analysis follows a *top-down approach*, starting with a *whole application* performance overview and then narrowing down to specific *hot spots*.
An initial overview can be obtained using Nsight Systems, either through the command line interface alone, or by complementing it with the provided GUI.

## Nsight Systems CLI

First, the benchmark application is compiled and executed to make sure the results are as expected.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v0.cpp -o ../build/stencil-2d-omp-target-v0
!../build/stencil-2d-omp-target-v0 double 8192 8192 2 256

Next, the binary is profiled with `nsys profile`.
The most important command line arguments are:
* `--stats=true`: prints a summary of performance statistics on the command line
* `-o ...`: sets the target output profile file
* `--force-overwrite=true`: replaces the profile file if it already exists (instead of aborting)

The collected data is configured with:
* `--trace=cuda`: logs CUDA API calls, host-device memory transfers, kernels, ...
* `--gpu-metrics-devices cuda-visible`: collects GPU metrics for all visible devices

A full list of available arguments for the profile mode is available as part of the official [documentation](https://docs.nvidia.com/nsight-systems/UserGuide/index.html#cli-profile-command-switch-options).

In [ ]:
!nsys profile --trace=cuda --gpu-metrics-devices cuda-visible --stats=true -o ../profiles/stencil-2d-omp-target-v0 --force-overwrite=true ../build/stencil-2d-omp-target-v0 double 8192 8192 2 256

The output of the command line is organized in multiple categories.
A possible output for an NVIDIA A100 (80 GB) is copied in below (truncated for conciseness and readability):

### Possible Output

```
[3/6] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)          Name        
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  -----------  --------------------
     53.5   23,873,092,559        516  46,265,683.3  48,966,433.5  42,596,053  50,987,472  3,207,513.1  cuMemcpyDtoHAsync_v2
     46.5   20,728,742,926        516  40,171,982.4  40,144,159.0  39,771,667  42,143,679    264,282.6  cuMemcpyHtoDAsync_v2
      0.0       11,123,053        258      43,112.6      40,565.0      28,256      76,902      8,066.8  cuLaunchKernel      
      0.0        1,858,042        258       7,201.7       7,059.0       5,842      17,876      1,151.6  cuStreamSynchronize 
      0.0        1,767,304          5     353,460.8     186,679.0      78,475     804,621    303,787.2  cuMemFree_v2        
      0.0        1,705,905          4     426,476.3     434,279.5      10,571     826,775    386,013.0  cuMemAlloc_v2       
      0.0        1,152,209          1   1,152,209.0   1,152,209.0   1,152,209   1,152,209          0.0  cuMemAllocHost_v2   
      0.0          667,360          1     667,360.0     667,360.0     667,360     667,360          0.0  cuMemFreeHost       
      0.0          581,831          1     581,831.0     581,831.0     581,831     581,831          0.0  cuModuleLoadDataEx  
      0.0          219,314          1     219,314.0     219,314.0     219,314     219,314          0.0  cuMemAllocManaged   
      0.0            6,853          5       1,370.6       1,152.0         211       2,695      1,155.2  cuCtxSetCurrent     
      0.0            1,924          1       1,924.0       1,924.0       1,924       1,924          0.0  cuInit              
```

```
[4/6] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                     Name                    
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  -------------------------------------------
    100.0    1,619,471,088        258  6,277,019.7  6,265,890.5  6,253,187  7,652,899    121,298.7  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14
```

```
[5/6] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  -----------  ----------------------------
     51.7   22,188,038,863    516  43,000,075.3  42,911,264.5  42,483,308  49,851,126    435,826.4  [CUDA memcpy Device-to-Host]
     48.3   20,709,165,277    516  40,134,041.2  40,108,880.5  39,738,994  42,107,502    265,040.6  [CUDA memcpy Host-to-Device]
```

```
[6/6] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)   Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 -----------  -----  --------  --------  --------  --------  -----------  ----------------------------
 277,025.391    516   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
 277,025.391    516   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

### Interpretation

The first wall of numbers is genuinely overwhelming: columns of nanoseconds and byte counts that mean nothing at first glance.
Fortunately the wise colleague is still within earshot, and together they sift the noise from the signal and pull out a handful of key insights.

Looking at the statistics they identify multiple effects:
* The number of kernel instances matches the expectation ($2 + 256$).
* The number of memory transfers seems to be related to the number of kernel instances.
* Comparing the time spent in GPU synchronization (`cuStreamSynchronize`) and kernel execution time shows a mismatch.
* Comparing aggregated memory transfer and kernel execution times reveals a stark difference.
  * \> Even if the kernel could be accelerated, overall performance will most likely not increase.
  * \> These numbers could be used to approximate a minimum number of iterations at which the memory transfers get amortized (assuming that the transfer times *don't scale with the number of iterations*).
* The size per transfer matches the expectation ($8192^2 \cdot 8 B \approx 537 MB$).

## Nsight Systems GUI

One thing nags at our PhD candidate: what *are* all those memory transfers doing there?
Numbers in a table only go so far, so they open the generated report in the graphical interface to see the timeline with their own eyes.

<div class="alert alert-block alert-info">

<b>Hands-On:</b>
Download the generated `stencil-2d-omp-target-v0.nsys-rep` report file directly from the `profiles` folder, or **shift + right-click** on this [link](../profiles/stencil-2d-omp-target-v0.nsys-rep) and select "Download".
Afterwards, open it in your local installation of Nsight Systems.

</div>

As might already be suspected, the timeline shows a recurring pattern of
* two memory transfers (HtoD),
* a kernel call, and
* two memory transfers (DtoH)

Additionally, `cudaStreamSynchronize` (and thereby `cuStreamSynchronize`) is only called at the end of this pattern, which partly explains the deviation from the kernel execution time.

At this stage, our PhD candidate has two options:
* Update the bottleneck model to explicitly include PCIe transfers, or
* Optimize the application to eliminate unnecessary data transfers.

The first option requires considering the peak bandwidth for the PCIe connection, which depends on the GPU generation:

|                                    |  Volta  | Ampere  | Hopper  | 
|------------------------------------|:-------:|:-------:|:-------:|
| PCIe                               | 3.0 x16 | 4.0 x16 | 5.0 x16 |
| Bandwidth (per direction) \[GB/s\] |  15.8   |  31.5   |  63.0   |

Our PhD candidate decides that explainable bad performance is better than inexplicable bad performance, but definitely less attractive than fixing the performance bug altogether.
They thus focus on reducing redundant transfers.

## Stencil Code Optimization - Data Transfers

Back to the course notes.
Buried in the OpenMP target offloading material, right where all important warnings live, i.e. in a box our PhD candidate skimmed the first time around, is the hint they needed all along: *minimize host-device data transfers; ideally move the data over once at the start and back once at the end.*

With the performance bug pinpointed, the data transfers can now be optimized.
One straightforward way is adding unstructured data primitives to the code, basically spanning a region at whose begin and end data is copied *only once*.
The updated version is available at [stencil-2d-omp-target-v1.cpp](../src/stencil-2d/stencil-2d-omp-target-v1.cpp), and can be compiled, executed and profiled using the following cells.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v1.cpp -o ../build/stencil-2d-omp-target-v1

In [ ]:
!../build/stencil-2d-omp-target-v1 double 8192 8192 2 256

In [ ]:
!nsys profile --trace=cuda --gpu-metrics-devices cuda-visible --stats=true -o ../profiles/stencil-2d-omp-target-v1 --force-overwrite=true ../build/stencil-2d-omp-target-v1 double 8192 8192 2 256

### Possible Output

```
[3/6] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Name        
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  ------------  --------------------
     89.2    1,651,659,215        260   6,352,535.4   6,279,550.5       3,096   7,676,825     682,218.3  cuStreamSynchronize 
      6.0      110,313,790          2  55,156,895.0  55,156,895.0  43,810,196  66,503,594  16,046,655.6  cuMemcpyDtoHAsync_v2
      4.4       81,729,351          2  40,864,675.5  40,864,675.5  40,354,992  41,374,359     720,801.3  cuMemcpyHtoDAsync_v2
      0.1        1,656,516          4     414,129.0     438,612.5      10,461     768,830     368,854.1  cuMemAlloc_v2       
      0.1        1,614,042          5     322,808.4     179,054.0      60,309     725,766     288,651.9  cuMemFree_v2        
      0.1        1,163,381          1   1,163,381.0   1,163,381.0   1,163,381   1,163,381           0.0  cuMemAllocHost_v2   
      0.1        1,082,324        258       4,195.1       3,667.0       3,236      45,760       3,123.4  cuLaunchKernel      
      0.0          680,356          1     680,356.0     680,356.0     680,356     680,356           0.0  cuMemFreeHost       
      0.0          559,196          1     559,196.0     559,196.0     559,196     559,196           0.0  cuModuleLoadDataEx  
      0.0          217,279          1     217,279.0     217,279.0     217,279     217,279           0.0  cuMemAllocManaged   
      0.0            5,612          5       1,122.4         972.0         220       2,736       1,005.2  cuCtxSetCurrent     
      0.0            2,034          1       2,034.0       2,034.0       2,034       2,034           0.0  cuInit              
```

```
[4/6] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                     Name                    
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  -------------------------------------------
    100.0    1,650,313,215        258  6,396,562.8  6,274,259.0  6,261,059  7,671,747    392,650.0  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14
```

```
[5/6] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)            Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  ------------  ----------------------------
     57.3      109,620,115      2  54,810,057.5  54,810,057.5  43,662,836  65,957,279  15,764,551.8  [CUDA memcpy Device-to-Host]
     42.7       81,604,965      2  40,802,482.5  40,802,482.5  40,303,730  41,301,235     705,342.5  [CUDA memcpy Host-to-Device]
```

```
[6/6] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)  Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 ----------  -----  --------  --------  --------  --------  -----------  ----------------------------
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

### Interpretation

Looking at the statistics, the effects previously discussed can now be compared:
* The number of kernel instances matches the expectation ($2 + 256$).
* ~~The number of memory transfers seems to be related to the number of kernel instances.~~
* ~~Comparing the time spent in GPU synchronization (`cuStreamSynchronize`) and kernel execution time shows a mismatch.~~
* ~~Comparing aggregated memory transfer and kernel execution times reveals an order of magnitude in difference.~~
* The size per transfer matches the expectation ($8192 \cdot 8192 \cdot 8 \text{B} \approx 537 \text{MB}$).

Opening the generated profile in the GUI reveals what was already expected: single staging parts at the beginning and end of the application, with multiple kernel calls in between.
The output file is once again collected in the `../profiles` folder and accessible via [link](../profiles/stencil-2d-omp-target-v1.nsys-rep).

Finally, a speed-up over the CPU baseline, and a real one this time.
Under normal circumstances this is the moment our PhD candidate would open a fresh document titled *paper_final_v1* and start typing.
But the performance model is still whispering: the code remains a comfortable factor away from the hardware limits.
And where there is a gap, there is a bigger, shinier speed-up waiting to be claimed.

*Sustained bandwidth in GB/s, as printed by the application.*

| Version | V100 | A100 | H100 |
| --- | ---: | ---: | ---: |
| `base` (serial CPU) | 10.8 | 24.4 | 22.2 |
| `omp-host` (OpenMP CPU) | 37.8 | 25.2 | 76.9 |
| `omp-target-v0` (OpenMP GPU, naive) | 2.3 | 6.3 | 10.2 |
| `omp-target-v1` (OpenMP GPU, fewer transfers) | 144.8 | 169.0 | 232.8 |

# Kernel Level Profiling

Another trip to the wise colleague's desk.
Their advice: now that the profiler confirms that *kernel execution* is the main hotspot, and no longer anything else such as data transfers, it is time to zoom in and put these kernels under the microscope.
The tool for that job is Nsight Compute.

The next step would usually be to use the application-level analysis to pinpoint the most critical kernel(s), i.e. those that contribute most to the overall execution time, and to focus on them first.
In this case, the step is straightforward: the application contains only a single kernel.

The kernel performance is profiled using the CLI of Nsight Compute: `ncu`.
Key command line arguments are (all command line arguments are listed in the [documentation](https://docs.nvidia.com/nsight-compute/NsightComputeCli/index.html#profile)):
* `-o ...`: sets the target output profile file (equivalent to `nsys`)
* `--force-overwrite`: replaces the profile file if it already exists (in contrast to `nsys` no `=true`)

By default *every kernel* is profiled.
Apart from adapting the number of iterations in the application, the scope of profiled kernels can be further limited with
* `--launch-skip n` or `-s n`: skips the first `n` kernels encountered
* `--launch-count n` or `-c n`: limits profiling to the first `n` applicable kernels
* `--kernel name` or `-k name`: limits profiling to kernels with the name `name`
  * can also be used with regex, e.g. `-k regex:"stencil[1-3]d"`
  * Nsight Compute also supports kernel renaming

In [ ]:
!ncu --launch-skip 2 --launch-count 1 -o ../profiles/stencil-2d-omp-target-v1 --force-overwrite ../build/stencil-2d-omp-target-v1 double 8192 8192 2 2

If no output file is specified, key profiling insights will be displayed directly in the command line output.

In [ ]:
!ncu --launch-skip 2 --launch-count 1 ../build/stencil-2d-omp-target-v1 double 8192 8192 2 2

### Possible Output

```
  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14 (64, 1, 1)x(128, 1, 1), Context 1, Stream 13, Device 0, CC 8.0
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         1.59
    SM Frequency                    Ghz         1.15
    Elapsed Cycles                cycle    8,848,896
    Memory Throughput                 %        35.24
    DRAM Throughput                   %         6.94
    Duration                         ms         7.66
    L1/TEX Cache Throughput           %        60.84
    L2 Cache Throughput               %        23.27
    SM Active Cycles              cycle 5,118,160.95
    Compute (SM) Throughput           %         1.10
    ----------------------- ----------- ------------

    OPT   This kernel grid is too small to fill the available resources on this device, resulting in only 0.05 full     
          waves across all SMs. Look at Launch Statistics for more details.                                             
```

```
    Section: Launch Statistics
    -------------------------------- --------------- ---------------
    Metric Name                          Metric Unit    Metric Value
    -------------------------------- --------------- ---------------
    Block Size                                                   128
    Function Cache Configuration                     CachePreferNone
    Grid Size                                                     64
    Registers Per Thread             register/thread              36
    Shared Memory Configuration Size           Kbyte           32.77
    Driver Shared Memory Per Block       Kbyte/block            1.02
    Dynamic Shared Memory Per Block       byte/block               0
    Static Shared Memory Per Block        byte/block               0
    # SMs                                         SM             108
    Stack Size                                                 1,024
    Threads                                   thread           8,192
    # TPCs                                                        54
    Enabled TPC IDs                                              all
    Uses Green Context                                             0
    Waves Per SM                                                0.05
    -------------------------------- --------------- ---------------

    OPT   Est. Speedup: 40.74%                                                                                          
          The grid for this launch is configured to execute only 64 blocks, which is less than the 108 multiprocessors  
          used. This can underutilize some multiprocessors. If you do not intend to execute this kernel concurrently    
          with other workloads, consider reducing the block size to have at least one block per multiprocessor or       
          increase the size of the grid to fully utilize the available hardware resources. See the Hardware Model       
          (https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-hw-model) description for more      
          details on launch configurations.                                                                             
```

```
    Section: Occupancy
    ------------------------------- ----------- ------------
    Metric Name                     Metric Unit Metric Value
    ------------------------------- ----------- ------------
    Block Limit SM                        block           32
    Block Limit Registers                 block           12
    Block Limit Shared Mem                block           32
    Block Limit Warps                     block           16
    Theoretical Active Warps per SM        warp           48
    Theoretical Occupancy                     %           75
    Achieved Occupancy                        %         6.25
    Achieved Active Warps Per SM           warp         4.00
    ------------------------------- ----------- ------------

    OPT   Est. Local Speedup: 91.67%                                                                                    
          The difference between calculated theoretical (75.0%) and measured achieved occupancy (6.3%) can be the       
          result of warp scheduling overheads or workload imbalances during the kernel execution. Load imbalances can   
          occur between warps within a block as well as across blocks of the same kernel. See the CUDA Best Practices   
          Guide (https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html#occupancy) for more details on     
          optimizing occupancy.                                                                                         
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Local Speedup: 25%                                                                                       
          The 12.00 theoretical warps per scheduler this kernel can issue according to its occupancy are below the      
          hardware maximum of 16. This kernel's theoretical occupancy (75.0%) is limited by the number of required      
          registers.                                                                                                    
```

```
    Section: GPU and Memory Workload Distribution
    -------------------------- ----------- -------------
    Metric Name                Metric Unit  Metric Value
    -------------------------- ----------- -------------
    Average DRAM Active Cycles       cycle    846,768.10
    Total DRAM Elapsed Cycles        cycle   488,183,296
    Average L1 Active Cycles         cycle  5,118,160.95
    Total L1 Elapsed Cycles          cycle   954,163,910
    Average L2 Active Cycles         cycle  8,363,718.94
    Total L2 Elapsed Cycles          cycle   671,136,080
    Average SM Active Cycles         cycle  5,118,160.95
    Total SM Elapsed Cycles          cycle   954,163,910
    Average SMSP Active Cycles       cycle  5,117,744.64
    Total SMSP Elapsed Cycles        cycle 3,816,655,640
    -------------------------- ----------- -------------

    OPT   Est. Speedup: 24.41%                                                                                          
          One or more SMs have a much lower number of active cycles than the average number of active cycles. Maximum   
          instance value is 42.14% above the average, while the minimum instance value is 100.00% below the average.    
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Speedup: 24.43%                                                                                          
          One or more SMSPs have a much lower number of active cycles than the average number of active cycles. Maximum 
          instance value is 42.17% above the average, while the minimum instance value is 100.00% below the average.    
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Speedup: 24.41%                                                                                          
          One or more L1 Slices have a much lower number of active cycles than the average number of active cycles.     
          Maximum instance value is 42.14% above the average, while the minimum instance value is 100.00% below the     
          average.                                                                                                      
```

### Interpretation

The first section, *GPU Speed Of Light Throughput*, provides a summary of key metrics and potential bottlenecks.
The most relevant ones here are:
* `Duration` of a single kernel execution (in ms)
* `DRAM Throughput` as a percentage of the theoretical maximum
* `Compute (SM) Throughput` as a percentage of the theoretical maximum

By default, Nsight Compute also offers suggestions for improving performance.
For example, the output might include:
```
OPT   This kernel grid is too small to fill the available resources on this device, resulting in only 0.05 full     
      waves across all SMs. Look at Launch Statistics for more details.                                             
```

Overwhelmed (again), our PhD candidate decides to simply start at the top of this list of helpful suggestions.
The very first one stops them cold: something about the grid being too small to fill the device with enough *waves*.
What on earth is a wave?
To make sense of it, and in general of what is happening on the GPU when work is offloaded, they set out to understand how a modern GPU is actually put together.

# GPU Architecture 101

## Example: NVIDIA H100

<img align="right" src="img/h100-chip.png" alt="H100 Chip" width="384px"/>

A fundamental design principle of modern GPUs is their hierarchical structure, which enables high levels of parallelism and scalability.
To better understand GPU performance, this section examines the NVIDIA H100 architecture as a representative example of state-of-the-art GPUs.

Further details can be found in the official [NVIDIA H100 White Paper](https://resources.nvidia.com/en-us-tensor-core).
The following figures and information are also derived from this resource.

![H100 Chip Layout with Annotations](img/h100-layout.png)

The diagram above depicts the 'full configuration' of a single NVIDIA H100 chip.
However, not all functional units shown are accessible in production GPUs.
For example, the SXM5 version of the H100 features **132 Streaming Multiprocessors (SMs)** enabled for computation.

<img align="right" src="img/h100-sm-layout.png" alt="H100 SM Layout" width="384px" style="background-color:white"/>

Each SM is subdivided into 4 sub partitions (SMSP), as visualized in the figure on the right, each with
* 16 INT32 units, for a total of $132 \cdot 4 \cdot 16 = 8448$,
* 32 FP32 units, for a total of $132 \cdot 4 \cdot 32 = 16896$,
* 16 FP64 units, for a total of $132 \cdot 4 \cdot 16 = 8448$, and
* 1 tensor core, for a total of $132 \cdot 4 = 528$.

Each unit is capable of executing one fused-multiply-add (FMA) operation per cycle, with the exception of the tensor cores which can each perform 512 FP16/FP32-mixed-precision FMAs at once.

## Memory Hierarchy

<img align="right" src="img/h100-memory-abstract.png" alt="memory abstraction" width="512px"/>

The GPU memory system is organized into a hierarchy designed to balance capacity, speed, and access latency.
Its levels, at the example of the H100, are (latency and bandwidth values provided by [GPU Benches](https://github.com/RRZE-HPC/gpu-benches))
- **Registers**
    - Scope: Private to each thread.
    - Latency: Typically 1 cycle.
    - Capacity: Each SM has 256 KB available for registers.
- **L1 Cache** and **Shared Memory**
    - Scope: Shared among all threads scheduled on a given SM. Threads of a given block can access the same shared memory.
    - Latency: ~32 cycles.
    - Capacity: Configurable up to 100 KB per SM for shared memory, with the remaining space used for L1 cache (up to 256 KB per SM).
- **L2 Cache**
    - Scope: Shared among all threads (GPU-wide)
    - Latency: ~280 cycles.
    - Capacity: 50 MB (two partitions with 25 MB each)
- **Global Memory (DRAM)**
    - Scope: Global to all threads.
    - Latency: ~690 cycles.
    - Capacity: between 80 and 94 GB
    - Bandwidth: between 2 and 3.9 TB/s.

## Waves and Occupancy

While this detour into GPU architecture was interesting, one open question still remains: what is a *wave*?

To answer this, it helps to clarify how threads are organized on GPUs:
- **OpenMP** organizes *threads* in *teams*.
- **CUDA** organizes *threads* in *blocks*.

Although OpenMP teams and CUDA thread blocks are not strictly equivalent, compilers typically map teams to blocks when targeting NVIDIA GPUs.
These blocks are then assigned to Streaming Multiprocessors (SMs):
* Each block is scheduled to one SM.
* Each SM can handle multiple blocks concurrently (depending on available resources).

Within a block, threads are further grouped into *warps*: groups of 32 threads.
Warps are the basic unit of scheduling on the SM sub-partitions (SMSPs).

A *wave* refers to the maximum number of blocks that can be resident on the GPU at the same time.
It is calculated as the number of SMs times the maximum number of blocks per SM.

Having at least one full wave ensures that all resources are fully utilized.
Multiple (full) waves work as well, but partial waves may lead to sub-optimal performance.

This closely relates to the concept of *occupancy* which is defined as the ratio of active warps to the maximum number of warps that can be scheduled on an SM.

# Micro Benchmarks

The architecture detour makes one thing clear: a GPU is hungry for parallelism, and lots of it.
But *how much* is enough?

Our PhD candidate faces two fundamental ways of approaching this question:

1. **Conceptual Analysis:** \
To have a GPU fully utilize its computational resources, it requires at least as many active threads as there are execution units for the current data type - typically on the order of ten thousand.
However, most applications require additional data transfers between main memory (or at least L2 cache) and the compute units, which introduces high latency.
To hide this latency and keep the GPU busy, resources need to be oversubscribed: when some warps are stalled waiting for data, others can be scheduled to execute.
In practice this usually relates to an order of magnitude more threads required (on the order of a hundred thousand).

2. **Benchmarking:** \
(Micro) benchmarks can help study performance for varying number of threads.
They can additionally be used to examine the effects of different data types, memory access patterns, computational intensities, the ratio of compute to data transfer, and more.

Rather than argue about it in the abstract, our PhD candidate decides to measure with a couple of micro-benchmarks they found on GitHub in the [Accelerated Programming EXamples (APEX)](https://github.com/SebastianKuckuk/apex) collection.
They choose two flavors: a [fused-multiply-add (FMA)](https://github.com/SebastianKuckuk/apex/tree/main/src/benchmark/fma) benchmark to assess computational throughput, and a [copy-stream](https://github.com/SebastianKuckuk/apex/tree/main/src/benchmark/stream) benchmark to evaluate main memory bandwidth.
Automated benchmarking of both is luckily available as part of the [APEX-Generator](https://github.com/SebastianKuckuk/apex-generator) project.

## FMA Benchmark

![FMA Benchmark Results for A100 80GB](img/A100-SXM4-80GB---fma.png)

The results for the FMA benchmark on a single A100 80GB reveal the following effects:
* The maximum performance observed is $9.7$ TFLOP/s.
    * This is pretty much the theoretical limit given by the data sheet for double precision.
* A distinct saw-tooth pattern is visible.
    * The first peak is at $26\,615$ threads, the first drop is at $28\,526$ threads. 
    * This is just around $256 \cdot 108 = 27\,648$ threads (using a block size of 256 threads, as chosen by the CUDA FMA benchmark), i.e. when each SM gets exactly one thread block.
    * At the first drop, performance nearly halves because the amount of work increases only slightly, but the execution time doubles due to one SM having doubled workload. 
* Compared to the CUDA version, the compiler seems to adapt block sizes when using OpenMP target offloading. 

## Stream Benchmark

![Stream Benchmark Results for A100 80GB](img/A100-SXM4-80GB---stream.png)

The results for the stream benchmark on a single A100 80GB reveal the following effects:
* The asymptotic bandwidth is around $1\,750$ GB/s, which is just around $85\%$ of the theoretical limit of $2\,040$ GB/s.
* The maximum bandwidth is around $2\,580$ GB/s for $1\,290\,948$ threads.
    * Each thread moves 16 bytes in the double precision case.
    * This corresponds to a total volume of just over $20$ MB, one L2 cache partition on the A100.
    * Since the dataset fits entirely into the L2 cache, the effective bandwidth exceeds the DRAM bandwidth limit and reflects L2 cache bandwidth instead.

## Stencil Code Optimization - Collapse

The benchmarks settle it: *tens of thousands* of threads are needed to saturate the compute units, and *hundreds of thousands* to saturate the memory system (without any other tricks or more complicated workloads per thread).
Glancing back at the stencil code in [stencil-2d-omp-target-v1](../src/stencil-2d/stencil-2d-omp-target-v1.cpp), our PhD candidate realizes they only parallelize the outer loop which only has 8190 iterations, nowhere near enough parallelism.

The OpenMP course suggested the proper fix: split the construct in two, mapping team parallelism to the outer loop and thread parallelism to the inner one.
That sounds like real work.
So they ask their favorite AI instead, and are cheerfully told about the `collapse` clause, which saves a lot of typing.
It is not *strictly* the same thing, as the AI neglects to dwell on and our PhD candidate declines to ask, but who really worries about the fine print?

The updated code is available at [stencil-2d-omp-target-v2](../src/stencil-2d/stencil-2d-omp-target-v2.cpp), and can be compiled, executed and profiled with the following cells.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v2.cpp -o ../build/stencil-2d-omp-target-v2

In [ ]:
!../build/stencil-2d-omp-target-v2 double 8192 8192 2 256

In [ ]:
!ncu --launch-skip 2 --launch-count 1 ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

### Possible Output

```
  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14 (524033, 1, 1)x(128, 1, 1), Context 1, Stream 13, Device 0, CC 8.0
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         1.59
    SM Frequency                    Ghz         1.15
    Elapsed Cycles                cycle    1,055,116
    Memory Throughput                 %        57.14
    DRAM Throughput                   %        57.14
    Duration                         us       913.66
    L1/TEX Cache Throughput           %        27.90
    L2 Cache Throughput               %        81.63
    SM Active Cycles              cycle 1,052,255.34
    Compute (SM) Throughput           %        64.41
    ----------------------- ----------- ------------

    INF   Compute and Memory are well-balanced: To reduce runtime, both computation and memory traffic must be reduced. 
          Check both the Compute Workload Analysis and Memory Workload Analysis sections.                               
```

### Evaluation

That already looks a lot better.
*Now* that is a speed-up worth writing up.

*Sustained bandwidth in GB/s, as printed by the application.*

| Version | V100 | A100 | H100 |
| --- | ---: | ---: | ---: |
| `base` (serial CPU) | 10.8 | 24.4 | 22.2 |
| `omp-host` (OpenMP CPU) | 37.8 | 25.2 | 76.9 |
| `omp-target-v0` (OpenMP GPU, naive) | 2.3 | 6.3 | 10.2 |
| `omp-target-v1` (OpenMP GPU, fewer transfers) | 144.8 | 169.0 | 232.8 |
| `omp-target-v2` (OpenMP GPU, collapse) | 620.0 | 1243.6 | 1662.3 |

Still, a nagging curiosity remains: how accurate are those *estimated* bandwidth numbers the code prints?
Nsight Compute reports percentages of peak, which is a start, but to turn a raw measurement into a percentage, it must already know the true, measured value.
Surely there is a way to ask it for that directly.
Digging through the public repository of a colleague who once did something similar, our PhD candidate finds exactly that: a way to make Nsight Compute report very specific pieces of information.

# Profiling Metrics

This approach looks very specific, our PhD candidate notes, and probably won't be useful for general performance analysis and engineering.
But for building automated pipelines that capture isolated performance metrics, this might be really helpful.
It also keeps the profiling overheads low.

The details are available for the [metrics structure](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-structure), and as a [metrics reference](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-reference), both part of NVIDIA's documentation.
`ncu --list-metrics` can also be used to obtain a list of available metrics - an example output is included in [profiles/metrics.txt](../profiles/metrics.txt).

The list is quite long and not particularly easy to navigate.
After a few hours of sifting through the list, our PhD candidate settles on the most relevant metrics for the application:
* `sm__warps_active.avg.pct_of_peak_sustained_active` displays the achieved occupancy.
* `dram__bytes_read` and `dram__bytes_write` correspond to the bytes read from and written to DRAM.
    This can be extended with `.sum` to obtain the total volume and `.sum.per_second` to obtain the bandwidth.
* `smsp__sass_thread_inst_executed_op_{dadd, dmul, dfma}_pred_on.sum` represents the total executed additions, multiplications and fused multiply-adds in double precision.
    The total number of FLOPs in double precision can be computed with `dadd + dmul + 2 * dfma`.
* `lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_{ld, st}.sum` can be used to query L2 load and store volumes.

Metrics to be measured can be provided to Nsight Compute on the command line using the optional argument `--metrics`:

In [ ]:
%%bash

ncu \
    --launch-skip 2 --launch-count 1 \
    --metrics \
sm__warps_active.avg.pct_of_peak_sustained_active,\
dram__bytes_read.sum,dram__bytes_write.sum,\
dram__bytes_read.sum.per_second,dram__bytes_write.sum.per_second,\
smsp__sass_thread_inst_executed_op_dadd_pred_on.sum,\
smsp__sass_thread_inst_executed_op_dmul_pred_on.sum,\
smsp__sass_thread_inst_executed_op_dfma_pred_on.sum,\
smsp__sass_thread_inst_executed_op_dadd_pred_on.sum.per_second,\
smsp__sass_thread_inst_executed_op_dmul_pred_on.sum.per_second,\
smsp__sass_thread_inst_executed_op_dfma_pred_on.sum.per_second\
    ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

### Possible Output

```
  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_14 (524033, 1, 1)x(128, 1, 1), Context 1, Stream 13, Device 0, CC 8.0
    Section: Command line profiler metrics
    -------------------------------------------------------------- ----------- ------------
    Metric Name                                                    Metric Unit Metric Value
    -------------------------------------------------------------- ----------- ------------
    dram__bytes_read.sum                                                 Mbyte       537.39
    dram__bytes_read.sum.per_second                                    Gbyte/s       588.07
    dram__bytes_write.sum                                                Mbyte       526.97
    dram__bytes_write.sum.per_second                                   Gbyte/s       576.67
    sm__warps_active.avg.pct_of_peak_sustained_active                        %        54.83
    smsp__sass_thread_inst_executed_op_dadd_pred_on.sum                   inst            0
    smsp__sass_thread_inst_executed_op_dadd_pred_on.sum.per_second      inst/s            0
    smsp__sass_thread_inst_executed_op_dfma_pred_on.sum                   inst  201,228,300
    smsp__sass_thread_inst_executed_op_dfma_pred_on.sum.per_second     inst/ns       220.20
    smsp__sass_thread_inst_executed_op_dmul_pred_on.sum                   inst   67,076,100
    smsp__sass_thread_inst_executed_op_dmul_pred_on.sum.per_second     inst/ns        73.40
    -------------------------------------------------------------- ----------- ------------
```

### General Guidelines

They also note down some general guidelines for using metrics:
* For counting metrics (e.g. bytes transferred) the structure is usually `metric.sum`.
* For corresponding rates this can be extended to `metric.sum.per_second`.
* Many metrics can be recomputed in terms of percentage of theoretical peak performance with `metric.sum.pct_of_peak_sustained_elapsed`.

## Nsight Compute Sets and Sections

Scrolling a little further in that same repository, they spot another convenience: *sets* and *sections*, a shorthand for requesting whole families of metrics at once, without typing out cryptic metric names one by one.

Providing a custom set of metrics can be quite useful, but also cumbersome.
A more coarse-grained way of telling Nsight Compute what to measure is using **sections** (examples of which have appeared before) or **sets**, both effectively pre-defined metric lists.

Available sections can be queried with `ncu --list-sections`, available sets with `ncu --list-sets`.
Exemplary outputs are available in [profiles/sections.txt](../profiles/sections.txt) and [profiles/sets.txt](../profiles/sets.txt).
To use them, additional flags are passed to Nsight Compute.

In [ ]:
!ncu --section=SpeedOfLight --launch-skip 2 --launch-count 1 ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

In [ ]:
!ncu --set=roofline --launch-skip 2 --launch-count 1 ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

# Nsight Compute GUI

The command line is unbeatable for automated, scripted profiling.
But our PhD candidate is more of a visual person, and squinting at ASCII tables is starting to wear thin.
So they do the obvious thing and open the profile data in the Nsight Compute GUI.\
They follow a similar pattern as before:
* Profiling of the application with suitable sections/sets remotely.
* Downloading the profile data.
* Opening it locally.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v2.cpp -o ../build/stencil-2d-omp-target-v2

In [ ]:
!../build/stencil-2d-omp-target-v2 double 8192 8192 2 256

In [ ]:
!ncu --set full --launch-skip 2 --launch-count 1 -o ../profiles/stencil-2d-omp-target-v2 --force-overwrite ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

<div class="alert alert-block alert-info">

<b>Hands-On:</b>
Download the obtained profile file `stencil-2d-omp-target-v2.ncu-rep` directly from the `profiles` folder, or by **shift + right-clicking** this [link](../profiles/stencil-2d-omp-target-v2.ncu-rep).
Afterwards, open it with your local installation of Nsight Compute.

</div>

<div class="alert alert-block alert-info">

<b>Hands-On:</b>
The volume of information is, once again, quite overwhelming when using the tool for the first time.
Start by double-clicking on (one of) the displayed kernels and take a moment to explore the different sections.
Expand them by clicking the little triangle next to them.
If you are unsure about certain terms, hover over them to see short documentation snippets as tooltips.

</div>

Our PhD candidate focuses on the **Memory Workload Analysis** section first.
It reveals that the data volume read from the L2 cache is quite high compared to the one from DRAM (about a factor of 3).

Why should the same computation move so much more data from L2 than from DRAM?
This is not a question a table will answer.
Our PhD candidate gets up, walks to the nearest whiteboard, and starts sketching how the threads actually touch memory.
On the right are the three schematic sketches they drew, showing the data read and written during stencil updates.

<img align="right" src="img/stencil-2d-single-update.png" alt="Single Stencil Update in a 2D Grid" width="300px"/>

First, a single stencil update is shown.
It is evident that one element is updated (written) and that this requires reading four neighbors.

<img align="right" src="img/stencil-2d-openmp.png" alt="Stencil Updates in a 2D Grid with a 1D Thread Decomposition" width="300px"/>

Following the parallelization employed by OpenMP results in reading about three times as many data elements as there are updates, or $3 \cdot blockSize + 2$.
This also corresponds to about three times as much data being read from L2 than written, which fits the measured values.

<img align="right" src="img/stencil-2d-cuda.png" alt="Stencil Updates in a 2D Grid with a 2D Thread Decomposition" width="300px"/>

To remedy this behavior, using (spatial) blocking techniques can be one option.
With this optimization, the volume of data read can then be approximated by $blockSize.x \cdot blockSize.y + 2 \cdot blockSize.x + 2 \cdot blockSize.y$.

# Stencil Code Optimization - Spatial Blocking

The sketch makes the diagnosis obvious: a 1D thread decomposition is a poor fit for a 2D stencil.
What is needed is *spatial blocking*.
This is achievable in OpenMP, by splitting the 2D loop into a 2D loop over blocks with 2D loops over threads inside, four nested loops in total, which, once again, sounds like a lot of work.

A colleague ambling past is drawn in by the tidy whiteboard art and asks what it is all about.
After a patient explanation, they simply shrug: *"Sounds complicated. I'd just write it in CUDA - you get a 2D thread decomposition, and the spatial blocking basically comes for free."*
Intrigued (and allergic to four nested loops), our PhD candidate decides to give CUDA a shot.

[stencil-2d-cuda-v3](../src/stencil-2d/stencil-2d-cuda-v3.cu) implements the approach detailed above.
As usual, the next cells can be used to compile and execute it.
Also note the switch from `nvc++` to `nvcc`.

In [ ]:
!nvcc -O3 -std=c++17 ../src/stencil-2d/stencil-2d-cuda-v3.cu -o ../build/stencil-2d-cuda-v3

In [ ]:
!../build/stencil-2d-cuda-v3 double 8192 8192 2 256

Oh no.
Not again.
The CUDA version is *way* too slow, and our PhD candidate is suddenly having vivid flashbacks to the opening act of this saga.
The prime suspect is the same as before, unnecessary data transfers, so, a little wiser now, they reach straight for the profiler instead of guessing.

*Sustained bandwidth in GB/s, as printed by the application.*

| Version | V100 | A100 | H100 |
| --- | ---: | ---: | ---: |
| `base` (serial CPU) | 10.8 | 24.4 | 22.2 |
| `omp-host` (OpenMP CPU) | 37.8 | 25.2 | 76.9 |
| `omp-target-v0` (OpenMP GPU, naive) | 2.3 | 6.3 | 10.2 |
| `omp-target-v1` (OpenMP GPU, fewer transfers) | 144.8 | 169.0 | 232.8 |
| `omp-target-v2` (OpenMP GPU, collapse) | 620.0 | 1243.6 | 1662.3 |
| `cuda-v3` (CUDA, spatial blocking) | 12.2 | 22.3 | 34.6 |

In [ ]:
!nsys profile --trace=cuda --gpu-metrics-devices cuda-visible --stats=true -o ../profiles/stencil-2d-cuda-v3 --force-overwrite=true ../build/stencil-2d-cuda-v3 double 8192 8192 2 256

### Possible Output

```
[3/6] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls     Avg (ns)         Med (ns)       Min (ns)       Max (ns)       StdDev (ns)             Name         
 --------  ---------------  ---------  ---------------  ---------------  -----------  --------------  ---------------  ----------------------
     92.9   12,442,234,177          2  6,221,117,088.5  6,221,117,088.5   96,885,097  12,345,349,080  8,660,971,941.5  cudaDeviceSynchronize 
      5.0      671,763,645          2    335,881,822.5    335,881,822.5  251,723,524     420,040,121    119,017,807.1  cudaMallocHost        
      1.4      191,407,420          2     95,703,710.0     95,703,710.0   95,620,786      95,786,634        117,272.2  cudaFreeHost          
      0.6       81,493,753          4     20,373,438.3     20,382,967.5   20,122,171      20,605,647        232,148.1  cudaMemcpy            
      0.0        2,132,268          1      2,132,268.0      2,132,268.0    2,132,268       2,132,268              0.0  cuLibraryLoadData     
      0.0        1,621,448          2        810,724.0        810,724.0      768,701         852,747         59,429.5  cudaFree              
      0.0        1,193,872          2        596,936.0        596,936.0      363,670         830,202        329,887.9  cudaMalloc            
      0.0        1,136,316        258          4,404.3          3,211.5        3,066         266,126         16,375.6  cudaLaunchKernel      
      0.0           29,623        258            114.8            110.0          100             922             55.5  cuKernelGetName       
      0.0            3,156          1          3,156.0          3,156.0        3,156           3,156              0.0  cuModuleGetLoadingMode
      0.0            1,192          1          1,192.0          1,192.0        1,192           1,192              0.0  cuLibraryGetKernel    
```

```
[4/6] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)                                   Name                                 
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  -----------  ----------------------------------------------------------------------
    100.0   12,442,900,559        258  48,228,296.7  48,290,464.0  47,029,086  49,685,855    549,898.6  void stencil2d<double>(const T1 *, T1 *, unsigned long, unsigned long)
```

```
[5/6] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  -----------  ----------------------------
     50.5       41,035,423      2  20,517,711.5  20,517,711.5  20,514,607  20,520,816      4,390.4  [CUDA memcpy Device-to-Host]
     49.5       40,209,273      2  20,104,636.5  20,104,636.5  20,103,532  20,105,741      1,562.0  [CUDA memcpy Host-to-Device]
```

```
[6/6] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)  Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 ----------  -----  --------  --------  --------  --------  -----------  ----------------------------
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

### Interpretation

Curious - the transfers are *not* the problem this time.
Whatever is going wrong must be happening *inside* the kernel itself, since every single one of its launches is now incredibly slow.
That is precisely the kind of question Nsight Compute exists to answer.

**Note:** `--set=full` is used here to capture as much data as possible.
Newer versions of Nsight Compute (>= 2026) capture *a lot* of data - use with care.

In [ ]:
!ncu --set full --launch-skip 2 --launch-count 1 -o ../profiles/stencil-2d-cuda-v3 --force-overwrite ../build/stencil-2d-cuda-v3 double 8192 8192 2 2

<div class="alert alert-block alert-info">

<b>Hands-On:</b>
Download the [report file](../profiles/stencil-2d-cuda-v3.ncu-rep) and open it locally.
To compare it against the last OpenMP profile, click `Compare` at the top right and then `Add Baseline`.
With the CUDA profile open, focus once again on the memory workload section.

</div>

Comparing the two profiles reveals multiple effects:
* The data volume read from L2 is now a lot lower (about 2x)
    - Cool, the optimization works.
* The data volume read from and written to DRAM is now about zero
    - This is ... *weird*.
* There is a new data path to system memory
    - Huh, that is definitely not good.

# Final Stencil Code Optimization

The previous analysis reports data transfers from and to system memory, or in other words the host memory, instead of using the GPU memory.
Reviewing the code reveals the culprit: instead of device pointers, host pointers are passed to the kernel.
In a sense, their intuition about shifting the bottleneck to the PCIe bus was correct - but not as bulk transfers which would show up in Nsight Systems, but rather as a massive number of accesses with tiny granularity.

Our PhD candidate is quietly relieved on two counts: first, that nobody witnessed this particular rookie mistake, and second, that the fix turns out to be delightfully simple.
The updated code version is available at [stencil-2d-cuda-v4](../src/stencil-2d/stencil-2d-cuda-v4.cu).

In [ ]:
!nvcc -O3 -std=c++17 ../src/stencil-2d/stencil-2d-cuda-v4.cu -o ../build/stencil-2d-cuda-v4

In [ ]:
!../build/stencil-2d-cuda-v4 double 8192 8192 2 256

The performance of the application is now quite close to the measured limit.
Further optimization of the *implementation* will likely not bring any performance benefits.
To further accelerate, *algorithmic optimizations* are required, e.g. more advanced (temporal) blocking or switching to a different numerical solver algorithm altogether.

*Sustained bandwidth in GB/s, as printed by the application.*

| Version | V100 | A100 | H100 |
| --- | ---: | ---: | ---: |
| `base` (serial CPU) | 10.8 | 24.4 | 22.2 |
| `omp-host` (OpenMP CPU) | 37.8 | 25.2 | 76.9 |
| `omp-target-v0` (OpenMP GPU, naive) | 2.3 | 6.3 | 10.2 |
| `omp-target-v1` (OpenMP GPU, fewer transfers) | 144.8 | 169.0 | 232.8 |
| `omp-target-v2` (OpenMP GPU, collapse) | 620.0 | 1243.6 | 1662.3 |
| `cuda-v3` (CUDA, spatial blocking) | 12.2 | 22.3 | 34.6 |
| `cuda-v4` (CUDA, device pointers) | 792.5 | 1526.1 | 1951.9 |

<div class="alert alert-block alert-info">

<b>Hands-On:</b>
The roofline section of the Nsight Compute profile shows traces of this.
Run the next cell, then download the profile [here](../profiles/stencil-2d-cuda-v4.ncu-rep) and review it in the **GPU Speed of Light Throughput** section by choosing **Roofline Double Precision** in the top-right drop-down.

</div>

In [ ]:
!ncu --set=full --launch-skip 2 --launch-count 1 -o ../profiles/stencil-2d-cuda-v4 --force-overwrite ../build/stencil-2d-cuda-v4 double 8192 8192 2 2

Roofline models are an extended version of the discussed bottleneck model.
Further details are beyond the scope of this introduction but, generally speaking: points in the chart that are close to the roofline indicate that the profiled code utilizes the hardware well.

<div class="alert alert-block alert-info">

<b>Hands-On:</b>
To complete the performance evaluation, also check the Nsight Systems timeline.
Run the cell below, then download the profile [here](../profiles/stencil-2d-cuda-v4.nsys-rep) and open it locally.

</div>

In [ ]:
!nsys profile --trace=cuda --gpu-metrics-devices cuda-visible --stats=true -o ../profiles/stencil-2d-cuda-v4 --force-overwrite=true ../build/stencil-2d-cuda-v4 double 8192 8192 2 256

### Possible Output

```
[3/6] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)       Med (ns)      Min (ns)     Max (ns)     StdDev (ns)            Name         
 --------  ---------------  ---------  -------------  -------------  -----------  -----------  -------------  ----------------------
     59.4      659,241,697          2  329,620,848.5  329,620,848.5  255,297,656  403,944,041  105,108,866.8  cudaMallocHost        
     16.3      181,064,100          2   90,532,050.0   90,532,050.0    1,383,729  179,680,371  126,074,764.6  cudaDeviceSynchronize 
     16.1      179,078,340          2   89,539,170.0   89,539,170.0   88,738,650   90,339,690    1,132,106.2  cudaFreeHost          
      7.4       82,145,819          4   20,536,454.8   20,540,788.5   20,119,118   20,945,124      337,439.3  cudaMemcpy            
      0.3        3,153,032          2    1,576,516.0    1,576,516.0    1,302,188    1,850,844      387,958.4  cudaMalloc            
      0.2        1,859,301          1    1,859,301.0    1,859,301.0    1,859,301    1,859,301            0.0  cuLibraryLoadData     
      0.1        1,569,938          2      784,969.0      784,969.0      781,176      788,762        5,364.1  cudaFree              
      0.1        1,147,112        258        4,446.2        3,146.0        3,046      272,910       16,817.8  cudaLaunchKernel      
      0.0           31,428        258          121.8          110.0          100          802           55.2  cuKernelGetName       
      0.0            3,467          1        3,467.0        3,467.0        3,467        3,467            0.0  cuModuleGetLoadingMode
      0.0            1,232          1        1,232.0        1,232.0        1,232        1,232            0.0  cuLibraryGetKernel    
```

```
[4/6] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances  Avg (ns)   Med (ns)   Min (ns)  Max (ns)  StdDev (ns)                                   Name                                 
 --------  ---------------  ---------  ---------  ---------  --------  --------  -----------  ----------------------------------------------------------------------
    100.0      181,771,389        258  704,540.3  704,322.0   692,513   717,090      4,517.5  void stencil2d<double>(const T1 *, T1 *, unsigned long, unsigned long)
```

```
[5/6] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  -----------  ----------------------------
     50.5       41,423,422      2  20,711,711.0  20,711,711.0  20,514,191  20,909,231    279,335.5  [CUDA memcpy Device-to-Host]
     49.5       40,531,132      2  20,265,566.0  20,265,566.0  20,103,469  20,427,663    229,239.8  [CUDA memcpy Host-to-Device]
```

```
[6/6] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)  Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 ----------  -----  --------  --------  --------  --------  -----------  ----------------------------
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

# Next Steps

And so, at last, our PhD candidate writes the paper, only to discover, at the worst possible moment, that stencil codes have been studied intensely for several decades and that a fast stencil kernel is, regrettably, not a novel contribution.
Squeezing out more performance now would require *algorithmic* changes, such as temporal blocking, diamond tiling, or a fundamentally more efficient solver algorithm, rather than more profiling.
Sensing that this is a problem for future-them, they close the laptop and go for a well-earned walk.

On that walk, our PhD candidate resolves to properly learn GPU performance analysis before wrestling with algorithmic optimizations.
The sections below collect what they picked up along the way: a short catalogue of common performance patterns to watch out for, and pointers to material they used to gain knowledge.

**Common Patterns**

In cases where performance is not close to theoretical limits there are often common patterns to be observed that provide an explanation:
* **Thread divergence**: threads of the same warp take different code paths.
* **(Un-)coalesced memory accesses**: consecutive threads access (non-)consecutive memory locations.
* **Kernel launch overhead**: many short-running kernels lead to a low ratio of time spent in kernels over total execution time.
* **Low occupancy**: lack of parallelism, and/or overusing available resources lead to a low occupancy.

**Additional Material**

* Many computing centers (regional, national and beyond) around the world offer excellent trainings, often online and free of charge for members of academia.
* The official NVIDIA documentation, in particular the [CUDA C++ Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/index.html) and the [CUDA C++ Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/), covers both the underlying concepts and the practical details.
* NVIDIA's [Nsight tools playlist](https://www.youtube.com/playlist?list=PL5B692fm6--ukF8S7ul5NmceZhXLRv_lR) walks through the profiling tools used in this tutorial in greater depth.
* [GTC](https://www.nvidia.com/gtc/) offers a wide spectrum of material around GPU computing. There is also an [on-demand archive](https://www.nvidia.com/on-demand/) with thousands of videos.